# Dock literature ligands into a target and look for clashes

Pick one or more crystal structures, list some ligands (SMILES, e.g. from a paper), run all cells. For each ligand and
target you get: the best docked pose, whether that pose **clashes** with the protein, whether it sits **in the site**,
and whether the docking protocol could be trusted on that target in the first place.

**What it answers:** *can this molecule be placed in this site without overlapping the protein?*
**What it does not answer:** how tightly it binds. Vina scores are shown, but in our SidA tests they are not a reliable
ranking (see "How to read the results"). The receptor is rigid, so a ligand that would need the protein to move will look worse than it is.

The worked example is **SidA** (*Aspergillus fumigatus*, L-ornithine hydroxylase): the substrate L-ornithine, two close analogues,
and one deliberately oversized molecule, docked into three crystal structures.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    import sidacore
except ImportError:                       # running from a clone that was not pip-installed
    sys.path.insert(0, str(Path.cwd().parent))
    import sidacore

import pandas as pd
from sidacore import screen as S

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", 30)

tools = S.check_tools()
missing = [k for k, v in tools.items() if v is None]
assert not missing, "missing programs: %s. Create the environment from environment.yml (see README.md)." % missing
print("programs found:", {k: Path(v).name for k, v in tools.items()})

## 2. Your parameters  (edit this cell, then run everything below)

**Targets.** One `TargetSpec` per structure:

| field | meaning |
|---|---|
| `pdb_id`, `chain` | the structure and the chain to use as the receptor |
| `site_ligand` | residue name of the ligand deposited in the site you care about. It centres the box and is redocked as a **control** |
| `keep` | `"auto"` keeps FAD / FDA / NADP when present (they line the SidA site); or a list, e.g. `["FAD"]` |
| `box_size` | edge of the search cube in Å. **12 = site-sized**: a ligand that does not fit is forced to clash. **22 = roomy**: an oversized ligand can dock clash-free at the rim, which is *not* a fit |
| `ph` | protonation of the protein (PROPKA) |

Waters, sulfate, glycerol and other crystallisation additives are always removed; the next section shows exactly what.

**Ligands.** A dict `{name: SMILES}`, or the path of a `.csv` (columns `name,smiles`) or `.smi` file. Write stereochemistry
(`[C@H]`) when it matters; the notebook warns when it is missing.

In [ ]:
# ---- targets -------------------------------------------------------------------------------------------
TARGETS = [
    S.TargetSpec("4B63", "A", site_ligand="ORN"),   # wild type, oxidised FAD, NADP+ and L-ornithine, 1.90 A
    S.TargetSpec("6X0J", "A", site_ligand="ORN"),   # reduced FAD (FDA), NADP and L-ornithine, chain A
    S.TargetSpec("6X0J", "C", site_ligand="ORN"),   # same structure, chain C
]

# ---- ligands -------------------------------------------------------------------------------------------
LIGANDS = {
    "L-ornithine (substrate)":            "NCCC[C@H](N)C(=O)O",
    "L-lysine":                           "NCCCC[C@H](N)C(=O)O",
    "L-arginine":                         "NC(N)=NCCC[C@H](N)C(=O)O",
    "tetraphenylmethane (too big on purpose)": "C(c1ccccc1)(c1ccccc1)(c1ccccc1)c1ccccc1",
}
# LIGANDS = "my_ligands.csv"        # <- or a file from a paper: columns name,smiles

# ---- settings ------------------------------------------------------------------------------------------
SETTINGS = S.Settings(
    ligand_ph=7.4,              # ligands are protonated at this pH (None = use the SMILES exactly as written)
    seeds=(1, 2, 3),            # docking repeats per ligand; more seeds = more reliable, slower
    exhaustiveness=16,          # Vina search effort
    num_modes=20,               # poses kept per run
    control_seeds=(1, 2, 3, 4, 5),   # repeats for the redock control of each target
)

# ---- where things go -----------------------------------------------------------------------------------
RUN_NAME = "sida_example"
WORK_DIR = Path("screen_runs") / RUN_NAME      # results.csv, controls.csv, poses/, prepared receptors
CACHE_DIR = Path("structures_cache")           # downloaded structures (reused)

## 3. What will be kept and removed

Check this before docking. **Cofactors that line the site must be kept**: removing FAD or NADP changes the pocket
(in SidA, NADP's ribose is 3.3 Å from the bound ornithine). Anything listed under *removed* is gone from the receptor.

In [ ]:
for spec in TARGETS:
    sel = S.preview_target(spec, CACHE_DIR)
    print("%s   box centre %s   cofactors kept: %s" % (spec.name, [round(float(x), 1) for x in sel.center], sel.kept_cofactors))
    print(pd.DataFrame(sel.removed).rename(columns={"min_dist_to_site": "closest to site (A)"}).to_string(index=False))
    for note in sel.notes:
        print("  NOTE:", note)
    print()

## 4. Run

For each target: prepare the receptor, **redock its own deposited ligand as a control**, then dock every ligand and measure clashes.
With the example (3 targets, 4 ligands) this takes a few minutes; the receptor preparation is cached, so a re-run is faster.

In [ ]:
import time
t0 = time.time()
result = S.run_screen(TARGETS, LIGANDS, settings=SETTINGS, cache_dir=CACHE_DIR, work_dir=WORK_DIR)
print("done in %.0f s; files in %s" % (time.time() - t0, WORK_DIR.resolve()))
for name, t in result.targets.items():
    for note in t.notes:
        print("NOTE %s: %s" % (name, note))

## 5. Can the protocol be trusted on each target?  (the control)

The deposited ligand is redocked into its own structure. If the protocol cannot put it back (**median top-1 RMSD above 2 Å**),
the target is marked `FAILED` and every result against it describes the *model*, not the molecule.
`crystal_ligand_clashes` is the clash count of the deposited pose itself; it should be 0.

**A pass is a modest statement.** The control uses the same box as your screen, and that box is centred on the deposited ligand, so the protocol is
told roughly where to look. A tighter box makes the redock easier: in our SidA runs 6X0J chain C fails with a 22 Å box (median top-1 RMSD 5.96 Å) but passes with
the 12 Å box used here (about 1 Å), while 4B63 fails at every box size from 12 to 22 Å. Read *validated* as "recovers the ligand when pointed at the site", not more.

In [ ]:
result.controls

## 6. Results

In [ ]:
cols = ["target", "ligand", "reading", "best_score", "n_clashes", "worst_clash", "n_clash_free", "n_poses",
        "centroid_dist", "site_contact_fraction", "net_charge", "warnings", "status"]
shown = result.results[[c for c in cols if c in result.results.columns]]
shown.round(2)

### How to read the results

* **`reading`** is the whole table in one phrase, by fixed rules: *clashes in every pose* (no docked pose is clash-free), *best pose clashes*,
  *clash-free but outside the site*, or *fits*. A prefix `(target failed its control)` means the target itself is not trustworthy.
* **`n_clashes`, `worst_clash`**: heavy-atom pairs in the best-scoring pose closer than the sum of van der Waals radii minus 0.4 Å
  (hydrogen-bond N/O pairs: closer than 2.4 Å). `n_clash_free` of `n_poses` asks whether *any* docked pose avoids the protein.
* **`centroid_dist`, `site_contact_fraction`**: is the pose in the deposited ligand's site? A pose can be clash-free simply because it sits at the
  rim of the box. `site_contact_fraction` is the share of the residues touching the deposited ligand that also touch the pose (1.0 = same neighbourhood).
* **`best_score`** (kcal/mol, more negative = better): shown because Vina gives it, **not a ranking**. In the SidA tests, with a 16-22 Å box a large
  hydrophobic molecule scored better than the real substrate because it could sit at the rim; and in our earlier 22 Å analysis of 4B63 the deposited
  ornithine pose scored about 0.4 kcal/mol *worse* than a wrong pose. Compare clashes and site occupancy, not scores.
* **The box matters.** Re-run with `box_size=22` to see an oversized ligand stop clashing and move to the rim; that is the box, not a fit.
  Conversely, a ligand that really extends beyond the crystal ligand's site will be penalised by a 12 Å box: compare box sizes before concluding.
* **Protonation matters.** SidA's δ-amine and the ligand pH are open questions; re-run with another `ligand_ph` to see whether a conclusion survives.

## 7. Look at a pose

Green = docked pose, grey sticks = the deposited ligand, red spheres = receptor atoms that clash.

In [ ]:
VIEW_TARGET = "6X0J_A"
VIEW_LIGAND = "tetraphenylmethane (too big on purpose)"

key = (VIEW_TARGET, VIEW_LIGAND)
S.show_pose(result.targets[VIEW_TARGET], result.poses[key], result.clashes[key]).show()

## 8. Clashes of the best pose, atom by atom

In [ ]:
rows = [{"target": t, "ligand": l, "clash": c.label()} for (t, l), cl in result.clashes.items() for c in cl[:5]]
pd.DataFrame(rows) if rows else print("no clashes in any best pose")

## 9. Compare with the reference run

`reference_results.csv` holds the table from the machine this notebook was built on (Linux, CPU Vina 1.2.7, the example parameters above, seeds 1-3).
If you changed any parameter above the comparison is meaningless. Vina is deterministic per seed on one machine; across machines small score differences are possible.

In [ ]:
ref_path = Path("reference_results.csv")
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    both = result.results.merge(ref, on=["target", "ligand"], suffixes=("", "_ref"))
    both["score_diff"] = both["best_score"] - both["best_score_ref"]
    print(both[["target", "ligand", "best_score", "best_score_ref", "score_diff", "n_clashes", "n_clashes_ref"]].round(2).to_string(index=False))
    print("\nlargest |score difference|: %.2f kcal/mol" % both["score_diff"].abs().max())
else:
    print("no reference_results.csv next to the notebook")

## Limits

* **The control is a weak test in a tight, ligand-centred box** (see section 5). A target that fails it is a real warning; a target that passes has only shown the redock is possible.
* **Rigid receptor, one conformation per structure.** Induced fit, loop movement and the FAD in/out switch are not modelled.
* **A clash is a geometric statement**, not a free energy. Thresholds (0.4 Å overlap, 2.4 Å for N/O pairs) were set from five deposited
  ornithine poses (worst non-polar overlap 0.16 Å, closest hydrogen bond 2.50 Å): a small calibration set.
* **Heavy atoms only**; hydrogens are not checked. Waters are removed, so water-mediated sites are not represented.
* **Only FAD, FDA and NADP can be kept as cofactors** (the only ones with a shipped template). Reduced NADPH (`NDP`), heme, FMN etc. are stripped and the list in section 3 says so.
* **Ligand preparation** takes the SMILES at face value after protonation; tautomers and ring conformers are not enumerated.